# 6. Quadratic programming with linear constraints  [CO 12.2, 14.6.1]

TOML-MIRI — Maximum Likelihood Estimation and Convex Optimization, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

np.set_printoptions(precision=4, suppress=True)
import cvxpy as cp
from scipy.optimize import minimize, LinearConstraint

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 6.1 A QP you can see

**Task.** Solve

$$\min_x\ \tfrac12x^\top Px + q^\top x \quad\text{s.t.}\quad Gx\le h, \qquad P\succ0,$$

in 2-D (choose $P$ with sliders as in Exercise 2.1 of the Linear Algebra portfolio, and a polygon $Gx\le h$). Plot the elliptic contours, the polygon, the unconstrained minimiser $-P^{-1}q$ and the constrained one.

Polygon: $x_1\ge-1$, $x_2\ge-1$, $x_1+x_2\le2$, $x_2-x_1\le2$ (a quadrilateral).

### Prediction

- If $-P^{-1}q$ is inside the polygon it is also the constrained optimum (no constraint active, all multipliers 0).
- Otherwise the optimum is on the boundary, where a contour ellipse just touches an edge (tangent) or sits at a vertex (two active constraints).

In [ ]:
Gm = np.array([[-1.0, 0.0], [0.0, -1.0], [1.0, 1.0], [-1.0, 1.0]])
hv = np.array([1.0, 1.0, 2.0, 2.0])
cnames = ["x₁ ≥ −1", "x₂ ≥ −1", "x₁ + x₂ ≤ 2", "x₂ − x₁ ≤ 2"]
poly = np.array([[-1, -1], [3, -1], [0, 2], [-1, 1]])          # its vertices, in order

def solve_qp(P, q):
    lam, U = np.linalg.eigh(P)
    L = np.diag(np.sqrt(np.clip(lam, 0, None))) @ U.T          # ½xᵀPx = ½‖Lx‖², valid also for PSD P
    x = cp.Variable(2)
    cons = [Gm @ x <= hv]
    prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(L @ x) + q @ x), cons)
    prob.solve()
    return x.value, cons[0].dual_value, prob.value

def plot_qp(a=2.0, b=0.5, c=1.0, q1=-4.0, q2=-3.0, ax=None):
    P = np.array([[a, b], [b, c]]); q = np.array([q1, q2])
    x_star, lam, val = solve_qp(P, q)
    ax = ax or plt.gca()
    g = np.linspace(-2.5, 3.5, 300); X1, X2 = np.meshgrid(g, g)
    F = 0.5 * (a * X1**2 + 2 * b * X1 * X2 + c * X2**2) + q1 * X1 + q2 * X2
    ax.contour(X1, X2, F, levels=np.sort(val + np.array([-3, -1, 0, 1, 3, 6, 10])), colors="gray", linewidths=0.7)
    ax.contour(X1, X2, F, levels=[val], colors="C3", linewidths=2)
    ax.fill(*poly.T, color="#cfe0f3", alpha=0.8)
    active = np.isclose(Gm @ x_star, hv, atol=1e-6)
    for i, (gi, hi) in enumerate(zip(Gm, hv)):
        if active[i]:
            t = np.linspace(-3, 4, 2)
            if abs(gi[1]) > 1e-12: ax.plot(t, (hi - gi[0] * t) / gi[1], "C2", lw=2.5)
            else: ax.axvline(hi / gi[0], color="C2", lw=2.5)
    if np.all(np.linalg.eigvalsh(P) > 1e-9):
        xu = -np.linalg.solve(P, q); ax.plot(*xu, "ko", ms=9, mfc="none", label=f"unconstrained −P⁻¹q {xu.round(2)}")
    ax.plot(*x_star, "r*", ms=16, label=f"constrained x* {x_star.round(3)}")
    grad = P @ x_star + q
    ax.annotate("", xy=x_star - 0.4 * grad / (np.linalg.norm(grad) + 1e-12), xytext=x_star,
                arrowprops=dict(arrowstyle="->", color="r", lw=2))
    ax.set_xlim(-2.5, 3.5); ax.set_ylim(-2.5, 3.5); ax.set_aspect("equal"); ax.legend(fontsize=8, loc="lower left")
    ax.set_title(f"eig(P) = {np.linalg.eigvalsh(P).round(2)};  active: {[n for n, a_ in zip(cnames, active) if a_]};  λ = {lam.round(3)}",
                 fontsize=9)
    return P, q, x_star, lam

plt.figure(figsize=(7.5, 7.5)); P, q, x_star, lam = plot_qp(); plt.show()

### Verify numerically: the KKT conditions at the solution

In [ ]:
check("primal feasibility Gx* ≤ h", np.all(Gm @ x_star <= hv + 1e-7))
check("dual feasibility λ ≥ 0", np.all(lam >= -1e-8))
check("stationarity Px* + q + Gᵀλ = 0", np.allclose(P @ x_star + q + Gm.T @ lam, 0, atol=1e-5))
check("complementary slackness λᵢ(Gx* − h)ᵢ = 0", np.allclose(lam * (Gm @ x_star - hv), 0, atol=1e-6))
sl = minimize(lambda x: 0.5 * x @ P @ x + q @ x, x0=np.zeros(2), method="SLSQP",
              constraints=[{"type": "ineq", "fun": lambda x: hv - Gm @ x}])
check("SLSQP finds the same x*", np.allclose(sl.x, x_star, atol=1e-4))

### Explore: move $q$ in and out; make $P$ only PSD

In [ ]:
@interact(a=FloatSlider(min=0, max=4, step=0.1, value=2.0), b=FloatSlider(min=-2, max=2, step=0.1, value=0.5),
          c=FloatSlider(min=0, max=4, step=0.1, value=1.0), q1=FloatSlider(min=-6, max=6, step=0.25, value=-4.0),
          q2=FloatSlider(min=-6, max=6, step=0.25, value=-3.0))
def explore_qp(a, b, c, q1, q2):
    if np.linalg.eigvalsh(np.array([[a, b], [b, c]])).min() < -1e-9:
        print("P is indefinite: not a convex QP — choose b² ≤ ac"); return
    plt.figure(figsize=(7.5, 7.5)); plot_qp(a, b, c, q1, q2); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7.5))
plot_qp(q1=-1.0, q2=-0.5, ax=axes[0]);  axes[0].set_title("q such that −P⁻¹q is inside: no active constraint\n" + axes[0].get_title(), fontsize=9)
plot_qp(q1=-6.0, q2=-6.0, ax=axes[1]);  axes[1].set_title("far outside: optimum on an edge or a vertex\n" + axes[1].get_title(), fontsize=9)
plot_qp(a=1.0, b=1.0, c=1.0, q1=-1.0, q2=0.0, ax=axes[2]); axes[2].set_title("P = [[1,1],[1,1]] only PSD: valley contours\n" + axes[2].get_title(), fontsize=9)
plt.show()
P_in, q_in = np.array([[2.0, 0.5], [0.5, 1.0]]), np.array([-1.0, -0.5])
x_in, lam_in, _ = solve_qp(P_in, q_in)
check("if −P⁻¹q is feasible, x* = −P⁻¹q and all λ = 0", np.allclose(x_in, -np.linalg.solve(P_in, q_in), atol=1e-5) and np.allclose(lam_in, 0, atol=1e-6))

---
### Explain

**1. When is the constraint irrelevant?**

When the unconstrained minimiser $-P^{-1}q$ already satisfies $Gx\le h$. Then it is feasible and globally optimal for the unconstrained problem, hence also for the constrained one; no constraint is active and all multipliers are 0 (KKT with $\lambda = 0$ is just $Px + q = 0$). Checked with $q = (-1, -0.5)$. More generally, an individual constraint is irrelevant whenever its multiplier is zero: removing it does not change the solution.

**2. When the optimum is on an edge, why is the contour tangent to it?**

At the optimum on edge $i$, stationarity gives $-\nabla f(x^\star) = \lambda_i g_i$ with $\lambda_i\ge0$: the steepest-descent direction points straight out of the polygon, along the edge's outward normal $g_i$. The gradient is normal to the contour through $x^\star$, so the contour and the edge share the same normal, i.e. they are tangent. Intuitively, if the contour crossed the edge transversally, part of the edge near $x^\star$ would lie inside a lower sublevel set, so sliding along the edge would decrease $f$, contradicting optimality. The red arrow ($-\nabla f$) in the plots is always perpendicular to the active edge.

**3. How many constraints can be active at once?**

In 2-D, generically **at most two**: two lines meet at a vertex, and $-\nabla f(x^\star)$ must then lie in the cone spanned by the two outward normals ($\lambda_1,\lambda_2\ge0$). A third line through the same point would be degenerate (as in the LP of 5.1). In general, in $\mathbb R^n$ up to $n$ linearly independent constraints can be active at a vertex. Unlike an LP, though, a QP optimum need not be at a vertex: it can be in the interior (0 active), on an edge (1 active) or at a vertex (2 active). With $P$ only PSD the contours become parallel lines (a valley); the optimum still exists because the polygon is bounded, but it may be non-unique along a direction of the null space of $P$ parallel to an edge.

---
## 6.2 Equality-constrained QP and the KKT system

**Task.** For random $P\succ0$ ($n = 5$) and $A\in\mathbb R^{2\times5}$, solve $\min\frac12x^\top Px + q^\top x$ s.t. $Ax = b$ by solving the KKT system and compare with `scipy.optimize.minimize` (method `trust-constr`).

$$\begin{bmatrix} P & A^\top\\ A & 0\end{bmatrix}\begin{bmatrix} x^\star\\ \mu^\star\end{bmatrix} = \begin{bmatrix} -q\\ b\end{bmatrix}.$$

### Prediction

- One linear solve gives the exact solution; trust-constr agrees to its tolerance.
- The KKT matrix is symmetric but indefinite: 5 positive and 2 negative eigenvalues (a saddle-point system).

In [ ]:
rng = np.random.default_rng(0)
n, p = 5, 2
B = rng.normal(size=(n, n)); P = B @ B.T + np.eye(n)
A = rng.normal(size=(p, n)); b = rng.normal(size=p); q = rng.normal(size=n)

K = np.block([[P, A.T], [A, np.zeros((p, p))]])
sol = np.linalg.solve(K, np.r_[-q, b])
x_kkt, mu_kkt = sol[:n], sol[n:]

tc = minimize(lambda x: 0.5 * x @ P @ x + q @ x, np.zeros(n), jac=lambda x: P @ x + q, hess=lambda x: P,
              method="trust-constr", constraints=[LinearConstraint(A, b, b)], options=dict(gtol=1e-10, xtol=1e-12))
print("x* (KKT)          =", x_kkt)
print("x* (trust-constr) =", tc.x)
print("μ* (KKT) =", mu_kkt, "   eig(KKT matrix) =", np.linalg.eigvalsh(K).round(3))

### Verify numerically

In [ ]:
check("KKT solution and trust-constr agree", np.allclose(x_kkt, tc.x, atol=1e-6))
check("primal feasibility Ax* = b", np.allclose(A @ x_kkt, b))
check("stationarity Px* + q + Aᵀμ* = 0", np.allclose(P @ x_kkt + q + A.T @ mu_kkt, 0))
check("KKT matrix is symmetric", np.allclose(K, K.T))
ev = np.linalg.eigvalsh(K)
check(f"KKT matrix is indefinite: {np.sum(ev > 0)} positive, {np.sum(ev < 0)} negative eigenvalues", np.sum(ev > 0) == n and np.sum(ev < 0) == p)
# P = 2I, q = 0: minimum-norm solution
K2 = np.block([[2 * np.eye(n), A.T], [A, np.zeros((p, p))]])
s2 = np.linalg.solve(K2, np.r_[np.zeros(n), b])
x_mn = A.T @ np.linalg.solve(A @ A.T, b)
check("P = 2I, q = 0: x* = Aᵀ(AAᵀ)⁻¹b (minimum-norm solution)", np.allclose(s2[:n], x_mn))
check("... = pinv(A) b", np.allclose(s2[:n], np.linalg.pinv(A) @ b))
check("... and μ* = −2(AAᵀ)⁻¹b", np.allclose(s2[n:], -2 * np.linalg.solve(A @ A.T, b)))
z = rng.normal(size=n); z_ns = z - np.linalg.pinv(A) @ (A @ z)          # a null-space direction
check("any other solution x* + z (Az = 0) has a larger norm", np.linalg.norm(x_mn + z_ns) > np.linalg.norm(x_mn))

---
### Explain

**1. Is the KKT matrix symmetric? Is it positive definite?**

**Symmetric: yes** — the off-diagonal blocks are $A^\top$ and $A$, and $P$ is symmetric. **Positive definite: no** — it is *indefinite*, with $n = 5$ positive and $p = 2$ negative eigenvalues (checked). The zero block already rules out PD: for $v = (0, w)$, $v^\top Kv = 0$. More precisely, it is congruent to $\mathrm{diag}(P, -AP^{-1}A^\top)$ (Schur complement), and $AP^{-1}A^\top\succ0$ when $A$ has full row rank, so there are exactly $p$ negative eigenvalues. This is the saddle-point structure of the Lagrangian: a minimum in $x$, a maximum in $\mu$. In practice, solve it with a symmetric-indefinite ($LDL^\top$) solver, not Cholesky. It is non-singular whenever $A$ has full row rank and $P$ is PD on the null space of $A$.

**2. With $P = 2I$ and $q = 0$, check that you recover the minimum-norm solution $A^\top(AA^\top)^{-1}b$ of [CO 14.3.2] (and Exercise 7.1 of the Linear Algebra portfolio).**

With $P = 2I$, $q = 0$ the objective is $x^\top x = \|x\|^2$, so the QP is "the smallest $x$ with $Ax = b$". KKT: $2x + A^\top\mu = 0\Rightarrow x = -\frac12A^\top\mu$, and $Ax = b\Rightarrow\mu = -2(AA^\top)^{-1}b$, so $x^\star = A^\top(AA^\top)^{-1}b = A^\dagger b$, checked numerically against both the formula and `pinv`. (The notes write $\mu^\star = -2(AA^\top)b$, missing the inverse.) Geometrically, as in LA 7.1: $x^\star = A^\top(\ldots)$ lies in the row space of $A$, and any other solution is $x^\star + z$ with $z\in\mathrm{NS}(A)\perp\mathrm{RS}(A)$, so $\|x^\star + z\|^2 = \|x^\star\|^2 + \|z\|^2 > \|x^\star\|^2$. The Lagrange multiplier simply encodes "$x$ must be a combination of the rows of $A$".